# PHẦN III — CONTROLLER VM TEMPLATE (RUNBOOK RÚT GỌN)

Notebook rút gọn: chỉ lệnh + comment ngắn.

## BƯỚC 12 — Download Ubuntu 24.04 Cloud Image

In [ ]:
sudo mkdir -p /var/lib/libvirt/templates
sudo chown -R $USER:$USER /var/lib/libvirt/templates
cd /var/lib/libvirt/templates

wget -c https://cloud-images.ubuntu.com/noble/current/noble-server-cloudimg-amd64.img -O ubuntu-24.04-base.img
wget -c https://cloud-images.ubuntu.com/noble/current/SHA256SUMS

sha256sum -c SHA256SUMS --ignore-missing  # expect OK
qemu-img info ubuntu-24.04-base.img

## BƯỚC 13 — Tạo Controller template

In [ ]:
cd /var/lib/libvirt/templates

cp ubuntu-24.04-base.img tpl-controller.img
qemu-img resize tpl-controller.img 100G

qemu-img info tpl-controller.img | grep "virtual size"
ls -lh tpl-controller.img

## BƯỚC 14 — virt-customize template

In [ ]:
cd /var/lib/libvirt/templates

sudo virt-customize -a tpl-controller.img \
  --update \
  --install curl,wget,git,vim,htop,net-tools,python3-pip,python3-venv,python3-dev,openssh-server,cloud-init,chrony,mariadb-client \
  --run-command "curl -fsSL https://get.docker.com | sh" \
  --run-command "systemctl enable ssh" \
  --run-command "systemctl enable docker" \
  --run-command "systemctl enable chrony" \
  --run-command "usermod -aG docker ubuntu" \
  --run-command "echo 'ubuntu ALL=(ALL) NOPASSWD:ALL' > /etc/sudoers.d/90-ubuntu-nopasswd" \
  --run-command "chmod 440 /etc/sudoers.d/90-ubuntu-nopasswd" \
  --run-command "python3 -m venv /opt/kolla-venv" \
  --run-command "/opt/kolla-venv/bin/pip install -U pip wheel" \
  --run-command "/opt/kolla-venv/bin/pip install 'ansible-core>=2.17,<2.18'" \
  --run-command "/opt/kolla-venv/bin/pip install 'kolla-ansible==2025.1.*'" \
  --run-command "/opt/kolla-venv/bin/kolla-ansible install-deps" \
  --timezone "Asia/Ho_Chi_Minh" 

## BƯỚC 14 — Verify template packages

In [ ]:
sudo virt-ls -a /var/lib/libvirt/templates/tpl-controller.img /opt/ | grep kolla-venv
sudo virt-cat -a /var/lib/libvirt/templates/tpl-controller.img /etc/sudoers.d/90-ubuntu-nopasswd
sudo virt-ls -a /var/lib/libvirt/templates/tpl-controller.img /etc/systemd/system/multi-user.target.wants/ | grep docker || true

## BƯỚC 15 — SSH password + inject key

In [ ]:
ssh-keygen -t ed25519 -f ~/.ssh/lab_instructor_key -C "instructor@lab" -N ""

cd /var/lib/libvirt/templates

sudo virt-customize -a tpl-controller.img \
  --run-command 'sed -i "s/^#PasswordAuthentication.*/PasswordAuthentication yes/" /etc/ssh/sshd_config' \
  --run-command 'sed -i "s/^PasswordAuthentication.*/PasswordAuthentication yes/" /etc/ssh/sshd_config' \
  --run-command 'sed -i "s/^#KbdInteractiveAuthentication.*/KbdInteractiveAuthentication yes/" /etc/ssh/sshd_config' \
  --run-command 'sed -i "s/^KbdInteractiveAuthentication.*/KbdInteractiveAuthentication yes/" /etc/ssh/sshd_config' \
  --run-command 'systemctl enable ssh' \
  --password "ubuntu:password:123.abc" \
  --root-password "password:123.abc" \
  --ssh-inject "ubuntu:file:$HOME/.ssh/lab_instructor_key.pub" 

## BƯỚC 15 — Verify SSH config

In [ ]:
sudo virt-cat -a /var/lib/libvirt/templates/tpl-controller.img /etc/ssh/sshd_config | grep -E "PasswordAuthentication|KbdInteractiveAuthentication"
sudo virt-cat -a /var/lib/libvirt/templates/tpl-controller.img /home/ubuntu/.ssh/authorized_keys || true
sudo virt-cat -a /var/lib/libvirt/templates/tpl-controller.img /etc/passwd | grep ubuntu

## BƯỚC 16 — Sysprep + lock template

In [ ]:
cd /var/lib/libvirt/templates

sudo virt-sysprep -a tpl-controller.img --operations defaults,-ssh-hostkeys
sudo virt-cat -a tpl-controller.img /etc/machine-id || true

sudo chmod 444 tpl-controller.img
ls -lh tpl-controller.img
qemu-img info tpl-controller.img